# Person 6 — Gradient Boosting & Deployment Reporting Pipeline

Pipeline Responsibility: Web App Deployment & Latency Benchmarking\nModel Assignment: Gradient Boosting Classifier

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root.")

ARTIFACTS = ROOT / "parts" / "artifacts"
OUTPUT_DIR = HERE / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

manifest = pd.read_csv(ARTIFACTS / "02_clean_manifest.csv")
features_data = np.load(ARTIFACTS / "03_features.npz")
X = features_data["X"]
y = manifest["label"].to_numpy()

dev_mask = manifest["split"] == "development"
test_mask = manifest["split"] == "test"

X_tr, y_tr = X[dev_mask], y[dev_mask]
X_te, y_te = X[test_mask], y[test_mask]

print(f"Loaded {len(X_tr)} training samples and {len(X_te)} test samples.")


Loaded 717 training samples and 180 test samples.


In [2]:
from sklearn.ensemble import GradientBoostingClassifier

print("--- Person 6: Gradient Boosting Model & Deployment Reporting ---")

gb_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=4,
    random_state=SEED
)

start_time = time.perf_counter()
gb_model.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time

preds = gb_model.predict(X_te)
acc = accuracy_score(y_te, preds)
p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average='macro')
cm = confusion_matrix(y_te, preds, labels=["Healthy", "Unhealthy"])

# Latency benchmark (per sample)
bench_start = time.perf_counter()
for _ in range(100):
    _ = gb_model.predict(X_te[:10])
latency_ms = ((time.perf_counter() - bench_start) / 1000.0) * 1000.0

print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
print(f"Average Latency: {latency_ms:.4f} ms / sample")
print("Confusion Matrix:\n", cm)

# Compile summary of all 6 models from their respective directories
model_summaries = []
models_info = [
    ("Logistic Regression", ROOT / "parts" / "logistic_regression" / "outputs" / "logistic_regression_metrics.json"),
    ("SVM", ROOT / "parts" / "svm" / "outputs" / "svm_metrics.json"),
    ("K-Nearest Neighbors", ROOT / "parts" / "knn" / "outputs" / "knn_metrics.json"),
    ("Decision Tree", ROOT / "parts" / "decision_tree" / "outputs" / "decision_tree_metrics.json"),
    ("Random Forest", ROOT / "parts" / "random_forest" / "outputs" / "random_forest_metrics.json"),
    ("Gradient Boosting", None)
]

gb_metrics_dict = {
    "model_name": "Gradient Boosting",
    "pipeline_stage": "Deployment & Benchmark Reporting",
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "precision": float(p),
    "recall": float(r),
    "fit_time_seconds": float(fit_time),
    "latency_ms_per_sample": float(latency_ms),
    "confusion_matrix": cm.tolist(),
    "classes": ["Healthy", "Unhealthy"]
}

with open(OUTPUT_DIR / "gradient_boosting_metrics.json", "w") as f:
    json.dump(gb_metrics_dict, f, indent=2)

joblib.dump(gb_model, OUTPUT_DIR / "gradient_boosting_model.joblib")

for name, path in models_info:
    if path and path.exists():
        with open(path) as f:
            m_data = json.load(f)
            model_summaries.append({
                "Model": m_data["model_name"],
                "Stage": m_data["pipeline_stage"],
                "Accuracy": m_data["accuracy"],
                "Macro_F1": m_data["macro_f1"],
                "Fit_Time_s": m_data["fit_time_seconds"]
            })
    elif name == "Gradient Boosting":
        model_summaries.append({
            "Model": "Gradient Boosting",
            "Stage": "Deployment & Benchmark Reporting",
            "Accuracy": float(acc),
            "Macro_F1": float(f1),
            "Fit_Time_s": float(fit_time)
        })

df_summary = pd.DataFrame(model_summaries)
df_summary.to_csv(OUTPUT_DIR / "model_comparison_6_members.csv", index=False)

print("\n--- 6-Member Model Comparison Table ---")
print(df_summary.to_string(index=False))


--- Person 6: Gradient Boosting Model & Deployment Reporting ---


Accuracy: 0.9722 | Macro F1: 0.9722
Average Latency: 0.0323 ms / sample
Confusion Matrix:
 [[92  0]
 [ 5 83]]

--- 6-Member Model Comparison Table ---
                          Model                              Stage  Accuracy  Macro_F1  Fit_Time_s
            Logistic Regression        Data Collection & Inventory  0.922222  0.922213    0.376914
               SVM (rbf kernel)         Data Preprocessing & Split  0.950000  0.949961    0.761120
      K-Nearest Neighbors (k=1)                Feature Engineering  0.950000  0.949961    0.020529
    Decision Tree (max_depth=7) Model Selection & Cross Validation  0.961111  0.960965    0.599810
Random Forest (Pipeline Winner)    Final Fit & Held-Out Evaluation  0.955556  0.955357    1.623137
              Gradient Boosting   Deployment & Benchmark Reporting  0.972222  0.972153   52.017846
